# Presentation du projet : solver quasi-lineaire pour l'equation de Westervelt

Ce notebook presente une version harmonisee avec l'implementation modulaire actuelle:
- equation quasi-lineaire,
- variable auxiliaire `F`,
- simulation 1D,
- snapshots,
- energie,
- mini etude de stabilite.


## Equation de Westervelt (forme quasi-lineaire)

On considere:
$$
\partial_t\!\left((1-2ku)u_t - b u_{xx}\right) - c^2 u_{xx} = 2k(u_t)^2.
$$

En introduisant la variable auxiliaire:
$$
F(u) = (1-2ku)u_t - b u_{xx},
$$

on obtient le schema discret:
$$
\begin{cases}
F_i^{n+1}=F_i^n+\Delta t c^2\delta_{xx}u_i^n,\\
\
 u_i^{n+1}=u_i^n+\Delta t\,\dfrac{F_i^{n+1}+b\,\delta_{xx}u_i^n}{1-2ku_i^n}.
\end{cases}
$$


In [ ]:
from src.solver import WesterveltSolver, WesterveltParams
from src.postprocessing import plot_stability_scan


## Configuration


In [ ]:
params = WesterveltParams(
    c=1500,
    epsilon=0.04,
    nu=1e-6,
    gamma=7,
    dx=1e-4,
    dt=2e-8,
    nx=1000,
    nt=1200,
    nonlinear=True,
    scheme="quasilinear",
)

solver = WesterveltSolver(params)
solver.initialize(u0_type="gaussian")


## Snapshots


In [ ]:
times = [0.0, 1e-6, 2e-6, 4e-6, 6e-6, 8e-6]
snapshots = solver.run_with_snapshots(times, store_energy=True)
solver.plot_snapshots(snapshots)


## Energie


In [ ]:
solver.plot_energy()


## Stabilite numerique (mini scan)


In [ ]:
dt_values = [1.2e-8, 1.8e-8, 2.5e-8, 3.0e-8]
amp_values = [0.5, 1.0, 1.5]

stability = solver.run_stability_scan(
    dt_values=dt_values,
    amplitude_values=amp_values,
    blowup_threshold=1e5,
)

stable_count = sum(1 for r in stability if r["stable"])
print(f"Configurations stables: {stable_count}/{len(stability)}")
plot_stability_scan(stability)


## Notes

- Changer `scheme="quasilinear"` en `scheme="explicit"` pour comparer les approches.
- Diminuer `nx` et `nt` pour des essais rapides.
- Surveiller la condition CFL affichee a l'initialisation.
